In [5]:
# ==============================================================================
# 1. SETUP E CARREGAMENTO
# ==============================================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import OneHotEncoder, TargetEncoder
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from boruta import BorutaPy
import optuna
import shap
from sklearn.metrics import classification_report, roc_auc_score, f1_score, precision_recall_curve

# Configurações de exibição
pd.set_option('display.max_columns', None)

# Carregamento dos dados (Assumindo que os arquivos estão no ambiente do Colab)
# O dataset principal unifica Sinistros, Veículos e Pessoas via 'id_sinistro' [cite: 1, 9, 15]
df_sinistros = pd.read_parquet('C:\\Users\\Matheus\\Desktop\\Arquivos_PUC\\sinistros_2022-2026.parquet')
df_veiculos = pd.read_parquet('C:\\Users\\Matheus\\Desktop\\Arquivos_PUC\\veiculos_2022-2026.parquet') # Carregar se necessário para merge
df_pessoas = pd.read_parquet('C:\\Users\\Matheus\\Desktop\\Arquivos_PUC\\pessoas_2022-2026.parquet')

# ==============================================================================
# 2. PRÉ-PROCESSAMENTO E TRATAMENTO DE LEAKAGE
# ==============================================================================

def preprocess_data(df):
    # Criar Variável Target: 1 para Fatal, 0 para Não Fatal 
    df['target'] = df['tipo_registro'].apply(lambda x: 1 if x == 'SINISTRO FATAL' else 0)
    
    # REMOÇÃO DE DATA LEAKAGE: Colunas que só existem após o óbito [cite: 17]
    leakage_cols = [
        'data_obito', 'ano_obito', 'mes_obito', 'dia_obito', 'ano_mes_obito', 
        'local_obito', 'tempo_sinistro_obito', 'qtd_gravidade_fatal', 
        'qtd_gravidade_grave', 'qtd_gravidade_leve'
    ]
    df = df.drop(columns=[c for c in leakage_cols if c in df.columns])
    
    # Drop de colunas administrativas/identificadores irrelevantes para o padrão estatístico
    df = df.drop(columns=['id_sinistro', 'tipo_registro', 'logradouro', 'numero_logradouro'], errors='ignore')
    
    # Tratamento de Nulos: Padronizar com 'NAO DISPONIVEL' conforme dicionário [cite: 5, 9, 16]
    cat_cols = df.select_dtypes(include=['object']).columns
    df[cat_cols] = df[cat_cols].fillna('NAO DISPONIVEL')
    
    return df

df_clean = preprocess_data(df_sinistros)

# ==============================================================================
# 3. FEATURE ENGINEERING
# ==============================================================================

# 3.1 Variáveis Cíclicas (Hora do Sinistro)
# Converte HH:MM para valor numérico e aplica Seno/Cosseno 
def encode_cyclic(df, col, max_val):
    df[f'{col}_sin'] = np.sin(2 * np.pi * df[col]/max_val)
    df[f'{col}_cos'] = np.cos(2 * np.pi * df[col]/max_val)
    return df

# Exemplo simplificado para o Mês 
df_clean = encode_cyclic(df_clean, 'mes_sinistro', 12)

# 3.2 Target Encoding para Alta Cardinalidade (Município)
encoder = TargetEncoder(smooth='auto')
df_clean['municipio_encoded'] = encoder.fit_transform(df_clean[['municipio']], df_clean['target'])

# ==============================================================================
# 4. TRATAMENTO DE ALTA CARDINALIDADE E DUMMIES (CORREÇÃO DE MEMÓRIA)
# ==============================================================================

# 1. Identificar colunas categóricas
cat_features = X.select_dtypes(include=['object', 'category']).columns.tolist()

# 2. Definir um limite (threshold) de cardinalidade
# Colunas com mais de 15 valores únicos serão tratadas com Target Encoding (ex: municipio, regiao)
# Colunas com 15 ou menos serão tratadas com One-Hot (Dummies)
high_card_cols = [col for col in cat_features if X[col].nunique() > 15]
low_card_cols = [col for col in cat_features if X[col].nunique() <= 15]

print(f"Alta Cardinalidade (Target Encoding): {high_card_cols}")
print(f"Baixa Cardinalidade (One-Hot): {low_card_cols}")

# 3. Aplicar Target Encoding nas colunas de alta cardinalidade
# Isso transforma o nome do município/via em um peso estatístico baseado na média da target
t_encoder = TargetEncoder(smooth='auto')
X[high_card_cols] = t_encoder.fit_transform(X[high_card_cols], y)

# 4. Aplicar One-Hot Encoding APENAS nas colunas de baixa cardinalidade
X_dummy = pd.get_dummies(X, columns=low_card_cols, drop_first=True)

# 5. Garantir que não existam IDs ou strings residuais que causem erro no Boruta
X_dummy = X_dummy.select_dtypes(exclude=['object'])

print(f"Nova forma da matriz: {X_dummy.shape}") # Deve ser drasticamente menor que 481k colunas

# ==============================================================================
# 4. SELEÇÃO DE FEATURES (BORUTA)
# ==============================================================================

X = df_clean.drop(columns=['target', 'municipio', 'data_sinistro', 'hora_sinistro', 'ano_mes_sinistro'], errors='ignore')
# Transformar as colunas categóricas restantes em dummies para o Boruta
X_dummy = pd.get_dummies(X, drop_first=True)
y = df_clean['target'].values

# Amostragem para agilizar o Boruta (devido ao prazo apertado)
X_sample, _, y_sample, _ = train_test_split(X_dummy, y, train_size=0.1, stratify=y, random_state=42)

rf = RandomForestClassifier(n_jobs=-1, class_weight='balanced', max_depth=5)
feat_selector = BorutaPy(rf, n_estimators='auto', verbose=2, random_state=42)
feat_selector.fit(X_sample.values, y_sample)

# Filtrar colunas selecionadas
selected_features = X_dummy.columns[feat_selector.support_].tolist()
X_final = X_dummy[selected_features]

# ==============================================================================
# 5. TREINAMENTO E TUNAGEM (OPTUNA)
# ==============================================================================

X_train, X_test, y_train, y_test = train_test_split(X_final, y, test_size=0.2, stratify=y, random_state=42)

def objective(trial):
    param = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'scale_pos_weight': (len(y_train) - sum(y_train)) / sum(y_train) # Trata desbalanceamento
    }
    model = XGBClassifier(**param)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    return f1_score(y_test, preds)

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=10) # Reduzido para 10 trials para cumprir o prazo de hoje

# Modelo Final
best_model = XGBClassifier(**study.best_params)
best_model.fit(X_train, y_train)

# ==============================================================================
# 6. INTERPRETABILIDADE (SHAP)
# ==============================================================================

explainer = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(X_test)

# Plot Global
shap.summary_plot(shap_values, X_test)

# ==============================================================================
# 7. AVALIAÇÃO FINAL
# ==============================================================================
y_pred = best_model.predict(X_test)
print(classification_report(y_test, y_pred))
print(f"ROC-AUC: {roc_auc_score(y_test, best_model.predict_proba(X_test)[:, 1]):.4f}")

Alta Cardinalidade (Target Encoding): ['latitude', 'longitude', 'regiao_administrativa', 'conservacao']
Baixa Cardinalidade (One-Hot): ['dia_da_semana', 'turno', 'tipo_via', 'tipo_local', 'administracao', 'circunscricao', 'tp_sinistro_primario', 'tp_sinistro_atropelamento', 'tp_sinistro_colisao_frontal', 'tp_sinistro_colisao_lateral', 'tp_sinistro_colisao_transversal', 'tp_sinistro_colisao_outros', 'tp_sinistro_choque', 'tp_sinistro_capotamento', 'tp_sinistro_engavetamento', 'tp_sinistro_tombamento', 'tp_sinistro_outros', 'tp_sinistro_nao_disponivel']


NameError: name 'y' is not defined